In [17]:
import pandas as pd

In [18]:
cust_info = pd.read_excel("Data/customer-info.xlsx")
credit_card = pd.read_excel("Data/credit-card-txns.xlsx")
debit_card = pd.read_excel("Data/debit-card-txns.xlsx")
digital_transact = pd.read_excel("Data/financial-digital-txns.xlsx")
instapay_incoming = pd.read_excel("Data/instapay-incoming-txns.xlsx")
instapay_outgoing = pd.read_excel("Data/instapay-outgoing-txns.xlsx")

df_dict = {
    'cust_info' : cust_info,
    'credit_card' : credit_card,
    'debit_card' : debit_card,
    'digital_transact' : digital_transact,
    'instapay_incoming' : instapay_incoming,
    'instapay_outgoing' : instapay_outgoing
}

In [19]:
for df_name, df in df_dict.items():
    df.rename(columns=str.lower, inplace=True)
    print(f'Column names in {df_name} : ')
    print(df.columns)
    print('-'*50)


Column names in cust_info : 
Index(['cust_num', 'province', 'region', 'education', 'age', 'tenure',
       'business_owner', 'digital_flag', 'income_source', 'idv_or_org',
       'segment', 'subsegment', 'gender', 'marital_status'],
      dtype='object')
--------------------------------------------------
Column names in credit_card : 
Index(['cust_num', 'merch_group', 'date', 'amount', 'channel', 'merch_name',
       'country'],
      dtype='object')
--------------------------------------------------
Column names in debit_card : 
Index(['cust_num', 'channel', 'date', 'time', 'amount'], dtype='object')
--------------------------------------------------
Column names in digital_transact : 
Index(['cust_num', 'amount', 'date_and_time', 'channel', 'tran_type'], dtype='object')
--------------------------------------------------
Column names in instapay_incoming : 
Index(['cust_num', 'date', 'time', 'amount', 'bpi_acct_type', 'src_bank_name'], dtype='object')
---------------------------------

In [20]:
possible_null_vals = [
    "NO_DATA", "N/A", "NA", "n/a", "na", "None", "none", "NONE",
    "null", "NULL", "Null", "-", "--", ".", "?",
    "Not Available", "NOT AVAILABLE", "missing", "MISSING", "",
    "0", "00", "000", "0000", "99", "999", "9999", "99999",
    "-1", "-999",
]

for df_name, df in df_dict.items():
    print(f'Dataframe {df_name}')
    print('-'*50)
    nulls_found = False

    for col in df.select_dtypes(include=['object','string']).columns:
        hits = df[col].isin(possible_null_vals)

        if hits.sum() > 0:
            nulls_found = True

            masked = df.loc[hits, col].value_counts()

            for val, count in masked.items():
                pct = count / len(df) * 100
                print(f'{col} : {val} appears {count} times ({pct:.2f}%)')
    
    print('\n')
        
    if not nulls_found:
        print('No placeholder nulls found.\n')

Dataframe cust_info
--------------------------------------------------
education : NO_DATA appears 39690 times (28.67%)
digital_flag : NO_DATA appears 41775 times (30.17%)
income_source : NO_DATA appears 1830 times (1.32%)


Dataframe credit_card
--------------------------------------------------


No placeholder nulls found.

Dataframe debit_card
--------------------------------------------------


No placeholder nulls found.

Dataframe digital_transact
--------------------------------------------------


No placeholder nulls found.

Dataframe instapay_incoming
--------------------------------------------------


No placeholder nulls found.

Dataframe instapay_outgoing
--------------------------------------------------


No placeholder nulls found.



In [21]:
# Rename the columns containing placeholder nulls into a new category called 'Unspecified'
cust_info['education'] = cust_info['education'].str.replace('NO_DATA','Unspecified')
cust_info['digital_flag'] = cust_info['digital_flag'].str.replace('NO_DATA','Unspecified')
cust_info['income_source'] = cust_info['income_source'].str.replace('NO_DATA','Unspecified')

# Cast the dtype of columns into the category dtype to save memory 
cust_info['education'] = cust_info['education'].astype('category')
cust_info['digital_flag'] = cust_info['digital_flag'].astype('category')
cust_info['income_source'] = cust_info['income_source'].astype('category')
cust_info['marital_status'] = cust_info['marital_status'].astype('category')

In [22]:
credit_card['merch_group'] = credit_card['merch_group'].fillna("Unspecified")
credit_card['country'] = credit_card['country'].fillna("Unspecified")

In [23]:
credit_card['date'] = pd.to_datetime(credit_card['date'])
debit_card['date'] = pd.to_datetime(debit_card['date'])
digital_transact['date_and_time'] = pd.to_datetime(digital_transact['date_and_time'])
instapay_incoming['date'] = pd.to_datetime(instapay_incoming['date'])
instapay_incoming = instapay_incoming.rename(columns={'bpi_acct_type':'acct_type'})
instapay_outgoing['date'] = pd.to_datetime(instapay_outgoing['date'])

In [24]:
for df_name, df in df_dict.items():
    duplicates = df.duplicated().sum()
    
    print(f"Duplicates in dataframe {df_name}: {duplicates} ({duplicates/len(df)*100:.2f}%)")

Duplicates in dataframe cust_info: 92294 (66.67%)
Duplicates in dataframe credit_card: 0 (0.00%)
Duplicates in dataframe debit_card: 0 (0.00%)
Duplicates in dataframe digital_transact: 0 (0.00%)
Duplicates in dataframe instapay_incoming: 0 (0.00%)
Duplicates in dataframe instapay_outgoing: 0 (0.00%)


In [25]:
digital_transact['date'] = digital_transact['date_and_time'].dt.date
digital_transact['time'] = digital_transact['date_and_time'].dt.time

In [26]:
digital_transact = digital_transact.drop(columns=['date_and_time'])

In [27]:
cust_info = cust_info.drop_duplicates()

In [28]:
# Save dataframes as parquets for further analysis
cust_info.to_parquet('Parquets/cust_info.parquet')
credit_card.to_parquet('Parquets/credit_card.parquet')
debit_card.to_parquet('Parquets/debit_card.parquet')
digital_transact.to_parquet('Parquets/digital_transact.parquet')
instapay_incoming.to_parquet('Parquets/instapay_incoming.parquet')
instapay_outgoing.to_parquet('Parquets/instapay_outgoing.parquet')